# Temporal Correlation Analysis: Tissue Turnover Rate vs. Age at Diagnosis
Testing Model 4 (Replication Stress-Dependent Haploinsufficiency) prediction that faster-dividing tissues cross the proofreading fidelity threshold first.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# Data from patient chronological timeline
data = pd.DataFrame({
    'finding': [
        'Colonic adenomas (first)',
        'Chronic gastric gastritis',
        'Endometriosis (symptomatic)',
        'Stage IV endometriosis (diagnosed)',
        'Benign breast tumor (PASH)',
        'Diffuse adenomyosis',
        'Thyroid carcinoma',
    ],
    'age_at_diagnosis': [19, 19, 22, 27, 27, 28, 28],
    'turnover_days': [4, 5, 28, 28, 60, 28, 2920],
    'tissue': [
        'Colonic epithelium',
        'Gastric epithelium', 
        'Endometrium',
        'Endometrium',
        'Breast stroma',
        'Endometrium/myometrium',
        'Thyroid epithelium',
    ],
    'phenotype_category': [
        'Neoplastic', 'Neoplastic', 'Endometrial/Hormonal',
        'Endometrial/Hormonal', 'Proliferative/Stromal',
        'Endometrial/Hormonal', 'Neoplastic',
    ]
})

# For correlation, use unique tissue types (deduplicate endometrium)
unique_tissues = data.groupby('tissue').agg({
    'age_at_diagnosis': 'min',  # first diagnosis in that tissue
    'turnover_days': 'first',
    'finding': 'first',
    'phenotype_category': 'first'
}).reset_index()

print(f"Total findings with known age: {len(data)}")
print(f"Unique tissue types: {len(unique_tissues)}")
print(unique_tissues[['tissue', 'turnover_days', 'age_at_diagnosis']].to_string(index=False))

## Spearman Rank Correlation
Spearman's rank correlation is appropriate here because: (1) small sample size (n<10), (2) we expect a monotonic but not necessarily linear relationship, (3) robust to outliers from the log-scale turnover rates.

In [ ]:
# Spearman rank correlation: turnover rate vs age at diagnosis
rho, p_value = stats.spearmanr(unique_tissues['turnover_days'], unique_tissues['age_at_diagnosis'])

print("=" * 60)
print("SPEARMAN RANK CORRELATION: Turnover Rate vs Age at Diagnosis")
print("=" * 60)
print(f"  Spearman rho:  {rho:.4f}")
print(f"  p-value:       {p_value:.4f}")
print(f"  n (tissues):   {len(unique_tissues)}")
print(f"  Direction:     {'Positive (longer turnover \u2192 later diagnosis)' if rho > 0 else 'Negative'}")
print(f"  Significance:  {'p < 0.05' if p_value < 0.05 else 'p >= 0.05 (not significant at alpha=0.05)'}")
print()

if rho > 0:
    print("INTERPRETATION: Positive correlation supports Model 4 prediction \u2014")
    print("tissues with faster cell division (shorter turnover period) are")
    print("diagnosed at younger ages.")
else:
    print("INTERPRETATION: No positive correlation observed.")

print()
print(f"Note: With n={len(unique_tissues)} observations, statistical power is limited.")
print("The biological plausibility and effect size are more informative than")
print("the p-value alone at this sample size.")

In [ ]:
# Pearson correlation on log-transformed turnover (tests log-linear relationship)
log_turnover = np.log10(unique_tissues['turnover_days'])
r_pearson, p_pearson = stats.pearsonr(log_turnover, unique_tissues['age_at_diagnosis'])

print("=" * 60)
print("PEARSON CORRELATION: log10(Turnover) vs Age at Diagnosis")
print("=" * 60)
print(f"  Pearson r:     {r_pearson:.4f}")
print(f"  R-squared:     {r_pearson**2:.4f}")
print(f"  p-value:       {p_pearson:.4f}")
print(f"  n (tissues):   {len(unique_tissues)}")
print()
print(f"R\u00b2 = {r_pearson**2:.2%} of variance in age-at-diagnosis is explained")
print(f"by log10(tissue turnover period).")

## Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

COLORS = {
    'Neoplastic': '#D32F2F',
    'Proliferative/Stromal': '#FF8F00',
    'Endometrial/Hormonal': '#7B1FA2',
}

# Panel A: Scatter with trend line
ax = axes[0]
for _, row in unique_tissues.iterrows():
    c = COLORS.get(row['phenotype_category'], '#78909C')
    ax.scatter(row['turnover_days'], row['age_at_diagnosis'],
               s=150, c=c, edgecolors='black', linewidth=0.8, alpha=0.85, zorder=3)
    ax.annotate(row['tissue'], xy=(row['turnover_days'], row['age_at_diagnosis']),
                xytext=(8, 4), textcoords='offset points', fontsize=7, color='#333')

# Log-linear trend line
log_t = np.log10(unique_tissues['turnover_days'])
coeffs = np.polyfit(log_t, unique_tissues['age_at_diagnosis'], 1)
x_trend = np.linspace(log_t.min() - 0.3, log_t.max() + 0.3, 100)
ax.plot(10**x_trend, np.polyval(coeffs, x_trend), 'r--', alpha=0.4, linewidth=1.5)

ax.set_xscale('log')
ax.set_xlabel('Cell Turnover Period (days, log scale)', fontweight='bold')
ax.set_ylabel('Age at First Diagnosis (years)', fontweight='bold')
ax.set_title(f'A. Turnover vs. Age at Diagnosis\n(Spearman \u03c1={rho:.2f}, p={p_value:.3f}; Pearson r={r_pearson:.2f})',
             fontsize=10, fontweight='bold')
ax.grid(alpha=0.2, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel B: Rank comparison
ax2 = axes[1]
turnover_rank = unique_tissues['turnover_days'].rank()
age_rank = unique_tissues['age_at_diagnosis'].rank()
for _, row in unique_tissues.iterrows():
    c = COLORS.get(row['phenotype_category'], '#78909C')
    t_rank = unique_tissues.loc[unique_tissues['tissue'] == row['tissue'], 'turnover_days'].rank().iloc[0]
    a_rank = unique_tissues.loc[unique_tissues['tissue'] == row['tissue'], 'age_at_diagnosis'].rank().iloc[0]
    idx = unique_tissues[unique_tissues['tissue'] == row['tissue']].index[0]
    ax2.scatter(turnover_rank.iloc[idx], age_rank.iloc[idx],
                s=150, c=c, edgecolors='black', linewidth=0.8, alpha=0.85, zorder=3)
    ax2.annotate(row['tissue'], xy=(turnover_rank.iloc[idx], age_rank.iloc[idx]),
                 xytext=(8, 4), textcoords='offset points', fontsize=7, color='#333')

# Perfect correlation line
max_rank = max(turnover_rank.max(), age_rank.max())
ax2.plot([0, max_rank + 1], [0, max_rank + 1], 'r--', alpha=0.3, linewidth=1)
ax2.set_xlabel('Turnover Period Rank (1=fastest)', fontweight='bold')
ax2.set_ylabel('Age at Diagnosis Rank (1=youngest)', fontweight='bold')
ax2.set_title('B. Rank Comparison\n(perfect correlation = diagonal)', fontsize=10, fontweight='bold')
ax2.grid(alpha=0.2, linestyle='--')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('temporal_correlation_analysis.png', dpi=300, bbox_inches='tight')
plt.savefig('temporal_correlation_analysis.svg', format='svg', bbox_inches='tight')
plt.show()
print("Saved: temporal_correlation_analysis.png/svg")

## Bootstrap Confidence Interval
Given the small sample size, bootstrap the Spearman correlation to estimate confidence intervals.

In [ ]:
np.random.seed(42)
n_bootstrap = 10000
n = len(unique_tissues)
bootstrap_rhos = []

for _ in range(n_bootstrap):
    idx = np.random.choice(n, size=n, replace=True)
    boot_turnover = unique_tissues['turnover_days'].iloc[idx].values
    boot_age = unique_tissues['age_at_diagnosis'].iloc[idx].values
    if len(set(boot_turnover)) > 1 and len(set(boot_age)) > 1:
        r, _ = stats.spearmanr(boot_turnover, boot_age)
        bootstrap_rhos.append(r)

bootstrap_rhos = np.array(bootstrap_rhos)
ci_low = np.percentile(bootstrap_rhos, 2.5)
ci_high = np.percentile(bootstrap_rhos, 97.5)

print("=" * 60)
print("BOOTSTRAP CONFIDENCE INTERVAL (10,000 resamples)")
print("=" * 60)
print(f"  Spearman rho:     {rho:.4f}")
print(f"  95% CI:           [{ci_low:.4f}, {ci_high:.4f}]")
print(f"  Bootstrap mean:   {bootstrap_rhos.mean():.4f}")
print(f"  Bootstrap median: {np.median(bootstrap_rhos):.4f}")
print()
if ci_low > 0:
    print("95% CI excludes zero \u2192 positive correlation is robust to resampling.")
else:
    print("95% CI includes zero \u2192 correlation direction uncertain with this sample size.")

## Summary and Model 4 Assessment

In [ ]:
print("=" * 70)
print("SUMMARY: TEMPORAL EVIDENCE FOR MODEL 4")
print("=" * 70)
print()
print("Model 4 (Replication Stress-Dependent Haploinsufficiency) predicts:")
print("  \u2192 Faster-dividing tissues cross the proofreading threshold first")
print("  \u2192 Age at clinical diagnosis should correlate with tissue turnover rate")
print()
print("Evidence:")
print(f"  1. Spearman \u03c1 = {rho:.3f} (p = {p_value:.4f})")
print(f"  2. Pearson r = {r_pearson:.3f} on log-transformed data (R\u00b2 = {r_pearson**2:.2%})")
print(f"  3. Bootstrap 95% CI for \u03c1: [{ci_low:.3f}, {ci_high:.3f}]")
print(f"  4. First non-congenital finding: colonic adenomas at age 19")
print(f"     (colonic epithelium has fastest turnover: 3-5 days)")
print(f"  5. Last diagnosed finding: thyroid carcinoma at age 28")
print(f"     (thyroid has slowest turnover: ~8 years)")
print(f"  6. Temporal gap: {28 - 19} years between fastest and slowest tissues")
print(f"     Turnover ratio: {2920/4:.0f}x difference in cell division rate")
print()
print("Conclusion: The temporal sequence of clinical findings is consistent")
print("with Model 4 predictions. While the small sample size limits")
print("statistical power, the biological plausibility, effect size, and")
print("directionality collectively support replication-coupled, dosage-")
print("dependent mutagenesis as a contributor to the multi-system phenotype.")

## Statistical Limitations

This analysis has important limitations that must be acknowledged:

1. **Small sample size (n=5-7 unique tissues):** Statistical power is limited. The correlation coefficient should be interpreted as hypothesis-generating, not confirmatory. A single outlier could dramatically change the result.

2. **Surveillance bias:** The colonic adenomas were detected at age 19 because colonoscopy was performed due to symptoms. The thyroid carcinoma was found incidentally on imaging. Earlier or later surveillance would shift the apparent age-at-onset without reflecting true biological latency.

3. **Age-dependent penetrance:** Some conditions (endometriosis, PASH) have natural age-of-onset distributions independent of POLE status. Endometriosis typically presents in the 20s-30s regardless of genetic background.

4. **Data point dependency:** Multiple endometrial findings (endometriosis symptomatic, Stage IV diagnosis, adenomyosis) represent the same tissue compartment. Deduplication partially addresses this but correlation remains influenced by how tissues are grouped.

5. **Turnover rate uncertainty:** Tissue turnover estimates are approximate and conflate different metrics (stem cell division rate vs. total tissue replacement time). Breast stromal turnover ("~60 days") is estimated and varies significantly with hormonal status.

6. **Confounding by hEDS:** The co-occurring hEDS could independently explain or modulate some findings (endometriosis severity, GI dysmotility), potentially altering the apparent tissue-turnover correlation.

**Interpretation guidance:** The positive correlation is consistent with Model 4 and should be viewed as supporting evidence alongside the biological plausibility argument, not as standalone statistical proof. Replication in a cohort of POLE carriers would be required for confirmatory evidence.